## 1) 평가의 개요
* 머신러닝은 데이터 가공/변환 -> 모델 학습/예측 -> 평가 프로세스를 가짐
* 성능 평가 지표 (evaluation metric)
  * 회귀 모델 : 대부분 실제값과 오차값의 오차 평균값에 기반. 예측오차를 가지고 정규화 수준을 재가공하는 방식
  * 분류 모델 : 일반적으로 실제 결과 데이터와 예측 결과 데이터가 얼마나 정확하고 오류가 적은가에 기반. 이진분류에서는 정확도보다 다른 지표가 더 중요시되기도 함
* 이번 장에서는 분류 모델의 성능 평가 지표에 중심을 둠
* 분류의 성능 평가 지표 : 정확도(accuracy) / 오차 행렬(confusion matrix) / 정밀도(precision) / 재현율(recall) / F1 스코어 / ROC AUC
* 분류는 결정 클래스 값 종류 유형에 따라 이진 분류(긍/부정 같은 2개의 결괏값만을 가지는 분류) / 멀티 분류(여러 개의 결정 클래스 값을 가지는 분류)

## 2) 정확도(accuracy)
* 실제 데이터에서 예측 데이터가 얼마나 같은지를 판단하는 지표
* 정확도 = (예측 결과가 동일한 데이터 건수)/ (전체 예측 데이터 건수)
* 직관적 모델 예측 성능을 나타내는 평가 지표
* 이진 분류의 경우 데이터 구성에 따라 ML 모델 성능 왜곡 가능성 있음(아래 예시) -> 정확도만으로 성능 평가 X
  * 특히 불균형한 레이블값 분포에서 ML 모델 성능 판단에서 왜곡 가능성 높음
  * 여러가지 분류 지표와 함께 적용해야 함

※ 사이킷런은 BaseEstimator 을 상속받으면 Customized 형태의 Estimator를 갭라자가 생성할 수 있음
※ 사이킷런은 load_digits() API 로 MNIST 데이터 세트를 제공

In [6]:
### 2.6. 참고

from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# 머신러닝 알고리즘에 불필요한 속성 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행
def format_features(df):
    df['Cabin']=df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [7]:
# BaseEstimator 클래스 상속받아 아무 학습 없이, 성별에 따라 생존자를 예측하는 단순 Classifier
from sklearn.base import BaseEstimator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

class MyDummyClassifier(BaseEstimator):
    def fit(self, x, y=None):
        pass # 학습을 수행하는 fit() 메서드는 아무것도 수행하지 않음
    
    def predict(self, x):
        # 예측을 수행하는 predict() 메서드는 단순히 성별이 1이면 0, 그렇지 않으면 1로 에측하는 classifier
        pred = np.zeros((x.shape[0], 1))
        for i in range(x.shape[0]):
            if x['Sex'].iloc[i] == 1:
                pred[i]=0
            else:
                pred[i]=1
        return pred
    
titanic_df = pd.read_csv(r'./kaggle/titanic/titanic_train.csv')
y_titanic_df = titanic_df['Survived']
x_titanic_df = titanic_df.drop('Survived', axis=1)
x_titanic_df = transform_features(x_titanic_df)
x_train, x_test, y_train, y_test = train_test_split(x_titanic_df, y_titanic_df, test_size=0.2, random_state=0)

myclf = MyDummyClassifier()
myclf.fit(x_train, y_train)

mypredictions = myclf.predict(x_test)
print('Dummy classifier 의 정확도 : {0:.4f}'.format(accuracy_score(y_test, mypredictions)))

Dummy classifier 의 정확도 : 0.7877
